<a href="https://colab.research.google.com/github/Necroline9520/PLANTILLA-PROY-2026/blob/main/Modelo_predictivo_final_jejeje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import optimizers
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, LSTM, RepeatVector, TimeDistributed, Flatten
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import plotly.offline as py #visualization
py.init_notebook_mode(connected=True) #visualization
import plotly.graph_objs as go #visualization
import plotly.tools as tls #visualization
import plotly.figure_factory as ff


from plotly.offline import init_notebook_mode, iplot

%matplotlib inline
warnings.filterwarnings("ignore")
init_notebook_mode(connected=True)

# Semillas para experimentos mas reproducibles.
from tensorflow.compat.v1 import set_random_seed
from numpy.random import seed
set_random_seed(1)
seed(1)

https://colab.research.google.com/github/Pradnya1208/Time-series-forecasting-using-Deep-Learning/blob/main/deep_learning_for_time_series_forecasting.ipynb

In [ ]:
import pandas as pd

# 1. Reemplaza con el nombre de tu archivo subido
# 2. Asegúrate de que 'date' sea el nombre de tu columna de fecha
ruta_archivo = '/content/tu_archivo.csv'

# Cargar tus datos
train_custom = pd.read_csv(ruta_archivo, parse_dates=['date'])

# Opcional: Renombrar columnas si tienen nombres diferentes
# train_custom = train_custom.rename(columns={'TuColumnaFecha': 'date', 'TuColumnaVentas': 'sales'})

print("Datos cargados correctamente:")
display(train_custom.head())

In [ ]:
# Instalar la librería openpyxl si no está instalada, necesaria para leer archivos .xlsx
!pip install openpyxl

### Descargar archivo Excel, convertir a CSV y cargar en Colab

Aquí debes proporcionar la URL directa del archivo Excel. Si la página de Odepa requiere alguna autenticación o interacción especial (como iniciar sesión, hacer clic en botones, etc.), este método directo podría no funcionar y necesitaríamos un enfoque diferente (por ejemplo, usar `selenium` para automatizar la navegación, lo cual es más complejo).

Para este ejemplo, asumimos que tienes una URL directa que descarga el archivo Excel.

In [ ]:
import requests
import pandas as pd
import os

# *** REEMPLAZA ESTA URL con el enlace de descarga directo de tu archivo Excel de Odepa ***
excel_url = 'https://www.odepa.gob.cl/wp-content/uploads/2026/04/Boletin-semanal-Afech_20260421.xlsx' # Reemplaza con la URL real

# Nombre del archivo temporal de Excel que se guardará en Colab
excel_filename = 'datos_odape.xlsx'
csv_filename = 'datos_odape.csv'

# Headers para simular una solicitud de navegador
headers = {
    'User-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# 1. Descargar el archivo Excel
try:
    print(f"Descargando archivo desde: {excel_url}")
    response = requests.get(excel_url, stream=True, headers=headers)
    response.raise_for_status() # Lanza un error para códigos de estado HTTP de clientes o servidores

    with open(excel_filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Archivo '{excel_filename}' descargado exitosamente.")

    # 2. Leer el archivo Excel y convertirlo a CSV
    # Los datos empiezan desde la fila 9 (header=8 en pandas)
    # y en las páginas (sheets) 2, 3 y 4 (índices 1, 2, 3 en pandas)
    xls = pd.ExcelFile(excel_filename)
    sheet_names_to_read = xls.sheet_names[1:4] # Obtener nombres de las hojas con índices 1, 2, 3

    all_sheets_df = []
    for sheet_name in sheet_names_to_read:
        df_temp = pd.read_excel(xls, sheet_name=sheet_name, header=8) # Leer cada hoja con el encabezado correcto
        all_sheets_df.append(df_temp)

    df_excel = pd.concat(all_sheets_df, ignore_index=True) # Concatenar todos los DataFrames

    # Opcional: Mostrar las columnas para verificar si se cargaron correctamente
    print("Columnas del DataFrame después de leer Excel y concatenar hojas:")
    print(df_excel.columns)
    display(df_excel.head()) # Mostrar el head después de concatenar

    df_excel.to_csv(csv_filename, index=False)
    print(f"Archivo convertido a '{csv_filename}'.")

    # Opcional: Eliminar el archivo Excel después de la conversión
    os.remove(excel_filename)
    print(f"Archivo Excel temporal '{excel_filename}' eliminado.")

    # 3. Cargar el CSV recién creado en un DataFrame de pandas
    df_data = pd.read_csv(csv_filename)
    print("Datos CSV cargados en DataFrame correctamente:")
    display(df_data.head())
    print(f"Forma del DataFrame: {df_data.shape}")

except requests.exceptions.RequestException as e:
    print(f"Error al descargar el archivo: {e}")
    print("Asegúrate de que la URL sea un enlace de descarga directo y esté correcta.")
except Exception as e:
    print(f"Ocurrió un error: {e}")
    print("Verifica que el archivo Excel tenga un formato válido, que la URL sea accesible y que el parámetro 'header' sea correcto.")

Descargando archivo desde: https://www.odepa.gob.cl/wp-content/uploads/2026/04/Boletin-semanal-Afech_20260421.xlsx
Archivo 'datos_odape.xlsx' descargado exitosamente.
Columnas del DataFrame después de leer Excel y concatenar hojas:
Index([       'Tattersall',         'Melipilla', 2026-04-20 00:00:00,
        3387.8835562549175,   2728.339049485546,   2408.634020618557,
        1893.0268199233717,  2892.7508361204013,   2348.789932236205,
        2812.3921271763816,  2690.9018302828617,   2187.602552415679,
         1263.979274611399,                   0,  1241.6870026525198,
                      2914,                2137,                1853,
                      1436,                2600,                1985,
         2437.260010468464,                2423,                1981,
                      1080,                1010,     'Agricultores ',
                 'Linares',                  56,                  32,
                        37,                   8,                  44

,Tattersall,Melipilla,2026-04-20 00:00:00,3387.883556,2728.339049,2408.634021,1893.02682,2892.750836,2348.789932,2812.392127,...,32,37,8,44,41,3,59,27,0.1,0.2
0,Tattersall,Curicó,2026-04-21,3143.210145,2865.187622,2361.847507,2535.138249,2725.642292,1353.151042,2551.055341,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Agricultores,Linares,2026-04-20,3095.704754,2937.166008,2241.743155,1608.386092,2789.046037,2742.044685,1811.519322,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Agricultores,Talca,2026-04-21,3284.130354,2951.575342,1839.000000,1593.225636,2624.716049,2720.866935,2252.125617,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Agricultores,Parral,2026-04-15,2909.907407,2971.176471,1927.090196,1316.217288,2906.049308,2722.546419,1881.480058,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Agricultores,San Javier,2026-04-17,2393.752711,2593.363559,1884.060023,1423.242706,2375.345732,2628.867133,1891.496656,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Archivo convertido a 'datos_odape.csv'.
Archivo Excel temporal 'datos_odape.xlsx' eliminado.
Datos CSV cargados en DataFrame correctamente:


,Tattersall,Melipilla,2026-04-20 00:00:00,3387.8835562549175,2728.339049485546,2408.634020618557,1893.0268199233717,2892.7508361204013,2348.789932236205,2812.3921271763816,...,32,37,8,44,41,3,59,27,0.1,0.2
0,Tattersall,Curicó,2026-04-21,3143.210145,2865.187622,2361.847507,2535.138249,2725.642292,1353.151042,2551.055341,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Agricultores,Linares,2026-04-20,3095.704754,2937.166008,2241.743155,1608.386092,2789.046037,2742.044685,1811.519322,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Agricultores,Talca,2026-04-21,3284.130354,2951.575342,1839.000000,1593.225636,2624.716049,2720.866935,2252.125617,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Agricultores,Parral,2026-04-15,2909.907407,2971.176471,1927.090196,1316.217288,2906.049308,2722.546419,1881.480058,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Agricultores,San Javier,2026-04-17,2393.752711,2593.363559,1884.060023,1423.242706,2375.345732,2628.867133,1891.496656,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Forma del DataFrame: (67, 39)


In [ ]:
import pandas as pd
import os
import re

DOWNLOAD_DIR = "odepa_data"

# Check if the DOWNLOAD_DIR exists. If not, instruct the user to run the download cell.
if not os.path.exists(DOWNLOAD_DIR):
    print(f"ERROR: El directorio '{DOWNLOAD_DIR}' no se encontró. Por favor, asegúrate de haber ejecutado la celda de descarga y conversión (celda `b54c6223`) antes de ejecutar esta celda.")
    # Exit or raise an error to prevent further execution if directory is missing
    raise FileNotFoundError(f"Directorio '{DOWNLOAD_DIR}' no encontrado. Ejecuta la celda de descarga.")


# Asumimos que el header_row_index es 7 para los archivos de 'Precio promedio'
# Este valor fue identificado en el paso anterior.
header_row_index = 7

# Definir el mapeo de nombres de columnas deseados (asumiendo orden posicional)
column_names_mapping = {
    'Unnamed: 0': 'Feria',
    'Unnamed: 1': 'Comuna',
    'Unnamed: 2': 'Fecha_Precio',
    'Unnamed: 3': 'Novillo_Gordo_Precio_kg',
    'Unnamed: 4': 'Novillo_Engorda_Precio_kg',
    'Unnamed: 5': 'Vaca_Gorda_Precio_kg',
    'Unnamed: 6': 'Vaca_Engorda_Precio_kg',
    'Unnamed: 7': 'Vaquilla_Gorda_Precio_kg',
    'Unnamed: 8': 'Vaquilla_Engorda_Precio_kg',
    'Unnamed: 9': 'Toros_Precio_kg',
    'Unnamed: 10': 'Terneros_Precio_kg',
    'Unnamed: 11': 'Terneras_Precio_kg',
    'Unnamed: 12': 'Cerdos_Precio_kg',
    'Unnamed: 13': 'Lanares_Precio_kg',
    'Unnamed: 14': 'Caballos_Precio_kg'
}

# Filtrar solo los archivos que contienen "Precio promedio" en el nombre de la hoja
csv_files_raw = [f for f in os.listdir(DOWNLOAD_DIR) if f.endswith('.csv')]
precio_promedio_csv_files = [f for f in csv_files_raw if 'Precio promedio' in f]

if precio_promedio_csv_files:
    # Función auxiliar para extraer la fecha del nombre del archivo
    def extract_date_from_filename(filename):
        match = re.search(r'(\d{4}\d{2}\d{2})', filename)
        if match:
            return pd.to_datetime(match.group(1), format='%Y%m%d', errors='coerce')
        return pd.Timestamp.min

    # Ordenar los archivos cronológicamente
    precio_promedio_csv_files.sort(key=extract_date_from_filename)

    # Lista para almacenar los DataFrames individuales
    df_list = []

    print(f"Cargando {len(precio_promedio_csv_files)} archivos de 'Precio promedio' con encabezado en la fila {header_row_index + 1}...")

    for file_name in precio_promedio_csv_files:
        file_path = os.path.join(DOWNLOAD_DIR, file_name)
        try:
            df_temp = pd.read_csv(file_path, header=header_row_index)

            # Limpieza básica de nombres de columnas (eliminar espacios y caracteres especiales)
            df_temp.columns = df_temp.columns.str.strip().str.replace(r'\s+', '_', regex=True).str.replace(r'[^\w_]', '', regex=True)

            # Construir la nueva lista de nombres de columnas en el orden correcto
            new_column_names = []
            for i in range(15): # Columnas Unnamed:0 a Unnamed:14
                original_col_key = f'Unnamed: {i}'
                new_column_names.append(column_names_mapping.get(original_col_key, f'Columna_{i}')) # Usar un fallback si falta en el mapeo

            # Verificar que el número de columnas coincide antes de renombrar
            if len(df_temp.columns) >= len(new_column_names):
                df_temp.columns = new_column_names + list(df_temp.columns[len(new_column_names):])
            else:
                print(f"ADVERTENCIA: Número de columnas en '{file_name}' no coincide con el mapeo. Renombrado parcial.")
                df_temp.columns = new_column_names[:len(df_temp.columns)]

            # Añadir una columna con la fecha del archivo (extraída del nombre)
            file_date = extract_date_from_filename(file_name)
            if pd.notna(file_date):
                df_temp['Fecha_Archivo'] = file_date

            # Convertir 'Fecha_Precio' a formato datetime, manejando errores
            df_temp['Fecha_Precio'] = pd.to_datetime(df_temp['Fecha_Precio'], errors='coerce')

            # Convertir columnas de precios a numéricas, manejando errores
            price_columns = [
                'Novillo_Gordo_Precio_kg',
                'Novillo_Engorda_Precio_kg',
                'Vaca_Gorda_Precio_kg',
                'Vaca_Engorda_Precio_kg',
                'Vaquilla_Gorda_Precio_kg',
                'Vaquilla_Engorda_Precio_kg',
                'Toros_Precio_kg',
                'Terneros_Precio_kg',
                'Terneras_Precio_kg',
                'Cerdos_Precio_kg',
                'Lanares_Precio_kg',
                'Caballos_Precio_kg'
            ]

            for col in price_columns:
                if col in df_temp.columns:
                    df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')

            df_list.append(df_temp)
        except Exception as e:
            print(f"ERROR al cargar '{file_name}': {e}")

    if df_list:
        # Concatenar todos los DataFrames en uno solo
        df_consolidado = pd.concat(df_list, ignore_index=True)
        print("\nDataFrame consolidado de 'Precio promedio' cargado y limpiado exitosamente:")
        display(df_consolidado.head())
        print("\nInformación general del DataFrame consolidado:")
        df_consolidado.info()
    else:
        print("No se pudo consolidar ningún DataFrame.")
else:
    print(f"No se encontraron archivos CSV de 'Precio promedio' en el directorio '{DOWNLOAD_DIR}'.")

ERROR: El directorio 'odepa_data' no se encontró. Por favor, asegúrate de haber ejecutado la celda de descarga y conversión (celda `b54c6223`) antes de ejecutar esta celda.


FileNotFoundError: Directorio 'odepa_data' no encontrado. Ejecuta la celda de descarga.

### Web Scraping para descargar mùltiples archivos Excel de Odepa

Vamos a usar `requests` para obtener el contenido de la página web y `BeautifulSoup` para analizar el HTML y extraer los enlaces a los archivos Excel.

In [ ]:
# Instalar BeautifulSoup si no está instalado
!pip install beautifulsoup4

In [4]:
import requests
from bs4 import BeautifulSoup
import os
import re
import pandas as pd

ODPA_BASE_URL = 'https://www.odepa.gob.cl'
WEBPAGE_URL = 'https://www.odepa.gob.cl/contenidos-rubro/boletines-del-rubro/boletin-semanal-de-precios-asoc-gremial-de-ferias-ganaderas'
DOWNLOAD_DIR = 'odepa_data'

# Crear el directorio de descarga si no existe
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

excel_links = []

# Headers para simular una solicitud de navegador
headers = {
    'User-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

try:
    print(f"Accediendo a la página: {WEBPAGE_URL}")
    # AÑADIR HEADERS a la solicitud para simular un navegador
    response = requests.get(WEBPAGE_URL, headers=headers)
    response.raise_for_status() # Lanza un error para códigos de estado HTTP de clientes o servidores

    soup = BeautifulSoup(response.text, 'html.parser')

    # Encontrar todos los enlaces que apuntan a archivos .xlsx
    # Odepa a menudo usa la clase 'pdf' o similar para los enlaces de descarga,
    # o simplemente busca el atributo href con '.xlsx'
    for link in soup.find_all('a', href=True):
        href = link['href']
        if href.endswith('.xlsx'):
            # Construir la URL completa si es relativa
            if href.startswith('/'):
                full_url = ODPA_BASE_URL + href
            else:
                full_url = href
            excel_links.append(full_url)

    print(f"Se encontraron {len(excel_links)} enlaces a archivos Excel.")
    # Eliminar duplicados si los hubiera
    excel_links = list(set(excel_links))
    excel_links.sort() # Ordenar para consistencia
    print("Enlaces Excel encontrados:")
    for link in excel_links:
        print(link)

except requests.exceptions.RequestException as e:
    print(f"Error al acceder a la página web: {e}")
except Exception as e:
    print(f"Ocurrió un error durante el scraping: {e}")

Accediendo a la página: https://www.odepa.gob.cl/contenidos-rubro/boletines-del-rubro/boletin-semanal-de-precios-asoc-gremial-de-ferias-ganaderas
Se encontraron 371 enlaces a archivos Excel.
Enlaces Excel encontrados:
https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6712/Boletin-semanal-Afech_31072019.xlsx
https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6713/Boletin-semanal-Afech_30052019.xlsx
https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6714/Boletin-semanal-Afech_22052019.xlsx
https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6715/Boletin-semanal-Afech_16052019.xlsx
https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6716/Boletin-semanal-Afech_09052019.xlsx
https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6717/Boletin-semanal-Afech_02052019.xlsx
https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6718/Boletin-semanal-Afech_25042019.xlsx
https://bibl

### Descargar y procesar cada archivo Excel

Ahora que tenemos los enlaces, vamos a iterar sobre ellos, descargar cada archivo, aplicar la misma lógica de lectura de hojas y encabezado que funcionó para un solo archivo, y luego consolidarlos.

In [ ]:
all_consolidated_dfs = []

for excel_url in excel_links:
    # Extraer el nombre del archivo de la URL
    file_name = os.path.join(DOWNLOAD_DIR, excel_url.split('/')[-1])

    print(f"\nProcesando: {excel_url}")

    try:
        # Descargar el archivo Excel
        response = requests.get(excel_url, stream=True, headers=headers)
        response.raise_for_status()

        with open(file_name, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Archivo '{os.path.basename(file_name)}' descargado exitosamente.")

        # Leer el archivo Excel y concatenar hojas
        xls = pd.ExcelFile(file_name)
        sheet_names_to_read = xls.sheet_names[1:4] # Hojas 2, 3 y 4 (indices 1, 2, 3)

        current_excel_dfs = []
        for sheet_name in sheet_names_to_read:
            # Asumiendo que los encabezados están en la fila 9 (index 8)
            df_temp = pd.read_excel(xls, sheet_name=sheet_name, header=8)
            current_excel_dfs.append(df_temp)

        if current_excel_dfs:
            df_excel_consolidated = pd.concat(current_excel_dfs, ignore_index=True)

            # Añadir una columna para identificar el archivo de origen si es ùtil
            df_excel_consolidated['Origen_Archivo'] = os.path.basename(file_name)

            all_consolidated_dfs.append(df_excel_consolidated)
            print(f"Datos de '{os.path.basename(file_name)}' procesados correctamente. Filas: {df_excel_consolidated.shape[0]}")
        else:
            print(f"No se pudieron procesar datos de '{os.path.basename(file_name)}' de las hojas especificadas.")

        # Opcional: Eliminar el archivo Excel después de procesarlo
        os.remove(file_name)

    except requests.exceptions.RequestException as e:
        print(f"Error al descargar/acceder a '{excel_url}': {e}")
    except Exception as e:
        print(f"Ocurrió un error al procesar '{os.path.basename(file_name)}': {e}")

if all_consolidated_dfs:
    df_data_all = pd.concat(all_consolidated_dfs, ignore_index=True)
    print("\n--- Consolidación Final ---")
    print(f"Todos los datos de Odepa han sido consolidados en un ùnico DataFrame con forma: {df_data_all.shape}")
    display(df_data_all.head())
    print("Información general del DataFrame consolidado:")
    df_data_all.info()
else:
    print("No se pudo consolidar ningùn DataFrame de los archivos descargados.")

# Limpiar el directorio de descarga si está vacío (opcional)
if not os.listdir(DOWNLOAD_DIR):
    os.rmdir(DOWNLOAD_DIR)
    print(f"Directorio '{DOWNLOAD_DIR}' eliminado.")


Procesando: https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6712/Boletin-semanal-Afech_31072019.xlsx
Error al descargar/acceder a 'https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6712/Boletin-semanal-Afech_31072019.xlsx': 404 Client Error: Not found for url: https://bibliotecadigital.odepa.gob.cl/bitstreams/8a5c9b7c-c44c-41b6-b598-b687262d708f/download

Procesando: https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6713/Boletin-semanal-Afech_30052019.xlsx
Archivo 'Boletin-semanal-Afech_30052019.xlsx' descargado exitosamente.
Datos de 'Boletin-semanal-Afech_30052019.xlsx' procesados correctamente. Filas: 73

Procesando: https://bibliotecadigital.odepa.gob.cl/bitstream/handle/20.500.12650/6714/Boletin-semanal-Afech_22052019.xlsx
Archivo 'Boletin-semanal-Afech_22052019.xlsx' descargado exitosamente.
Datos de 'Boletin-semanal-Afech_22052019.xlsx' procesados correctamente. Filas: 73

Procesando: https://bibliotecadigital.odepa.gob

### Separar los datos en conjuntos de entrenamiento y prueba

Ahora que `df_data_all` contiene todos los datos consolidados, podemos usar `train_test_split` de `sklearn` para dividirlos en conjuntos de entrenamiento y prueba, y guardarlos en directorios separados segùn tu solicitud.

In [ ]:
# @title
from sklearn.model_selection import train_test_split
import os

# Crear directorios para los conjuntos de entrenamiento y prueba
TRAIN_DIR = 'data_train'
TEST_DIR = 'data_test'
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# Asumiendo que 'df_data_all' es el DataFrame consolidado de todos los archivos

# Dividir los datos en entrenamiento y prueba
# Puedes ajustar 'test_size' (proporción para el conjunto de prueba) y 'random_state' (para reproducibilidad)

train_set, test_set = train_test_split(df_data_all, test_size=0.2, random_state=42)

print("Datos divididos en entrenamiento y prueba:")
print(f"Forma del conjunto de entrenamiento: {train_set.shape}")
print(f"Forma del conjunto de prueba: {test_set.shape}")

# Guardar los conjuntos en archivos CSV separados dentro de sus respectivos directorios
train_file_path = os.path.join(TRAIN_DIR, 'train_data.csv')
test_file_path = os.path.join(TEST_DIR, 'test_data.csv')

train_set.to_csv(train_file_path, index=False)
test_set.to_csv(test_file_path, index=False)

print(f"Conjunto de entrenamiento guardado en: {train_file_path}")
print(f"Conjunto de prueba guardado en: {test_file_path}")

print("\nPrimeras 5 filas del conjunto de entrenamiento:")
display(train_set.head())

print("\nPrimeras 5 filas del conjunto de prueba:")
display(test_set.head())

NameError: name 'df_data_all' is not defined

### Separar los datos en conjuntos de entrenamiento y prueba

Ahora que tenemos los datos cargados en el DataFrame `df_data`, podemos usar `train_test_split` de `sklearn` para dividirlos.

In [ ]:
from sklearn.model_selection import train_test_split

# Asumiendo que 'df_data' es el DataFrame que acabamos de cargar

# Dividir los datos en entrenamiento y prueba
# Puedes ajustar 'test_size' (proporción para el conjunto de prueba) y 'random_state' (para reproducibilidad)

train_set, test_set = train_test_split(df_data, test_size=0.2, random_state=42)

print("Datos divididos en entrenamiento y prueba:")
print(f"Forma del conjunto de entrenamiento: {train_set.shape}")
print(f"Forma del conjunto de prueba: {test_set.shape}")

print("\nPrimeras 5 filas del conjunto de entrenamiento:")
display(train_set.head())

print("\nPrimeras 5 filas del conjunto de prueba:")
display(test_set.head())

NameError: name 'df_data' is not defined